<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum/temp_work_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import duckdb

with duckdb.connect("nse_200_data.duckdb") as con:
    df = con.execute("SELECT * FROM equities").fetchdf()

print(df.shape)

(2509153, 14)


In [4]:
df.columns

Index(['SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST', 'PREVCLOSE',
       'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP', 'TOTALTRADES', 'ISIN',
       'Unnamed: 13'],
      dtype='object')

In [5]:
df.tail(5)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN,Unnamed: 13
2509148,ZTECH,SM,285.90,285.90,281.35,285.90,285.90,272.30,285600,8.162346e+07,05-JUL-2024,133,INE0ISZ01012,NaN
2509149,ZUARI,EQ,223.10,225.15,220.25,221.20,222.95,223.10,220298,4.903669e+07,05-JUL-2024,8248,INE840M01016,NaN
2509150,ZUARIIND,EQ,396.05,398.50,384.10,388.20,388.55,396.10,162292,6.353563e+07,05-JUL-2024,9678,INE217A01012,NaN
2509151,ZYDUSLIFE,EQ,1145.00,1166.85,1137.05,1162.45,1161.00,1137.05,2734011,3.163353e+09,05-JUL-2024,105113,INE010B01027,NaN
2509152,ZYDUSWELL,EQ,2081.05,2135.00,2072.00,2118.25,2125.00,2097.90,49564,1.038657e+08,05-JUL-2024,6912,INE768C01010,NaN


In [ ]:
import os
import zipfile
import shutil
import pandas as pd
import numpy as np

source_folder = "/content"           # Folder containing ZIP files
levelone_folder = "/content/levelone"

os.makedirs(levelone_folder, exist_ok=True)

def extract_all_zips(source, destination):

    # First copy all zip files to levelone
    for file in os.listdir(source):
        if file.endswith(".zip"):
            shutil.copy(os.path.join(source, file), destination)

    # Now recursively extract inside levelone
    while True:
        zip_found = False

        for root, dirs, files in os.walk(destination):
            for file in files:
                if file.endswith(".zip"):
                    zip_found = True
                    zip_path = os.path.join(root, file)

                    with zipfile.ZipFile(zip_path, 'r') as z:
                        z.extractall(root)

                    os.remove(zip_path)
                    print(f"Extracted and removed: {file}")

        if not zip_found:
            break

    print("All ZIP files fully extracted into levelone.")


extract_all_zips(source_folder, levelone_folder)

In [13]:
import pandas as pd
import glob
import os

# Path to folder containing CSV files
folder_path = "/content/levelone"

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and append
df_list = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

# Combine all into one dataframe
full_data = pd.concat(df_list, ignore_index=True)

In [14]:
full_data.columns

Index(['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN',
       'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric',
       'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric',
       'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst',
       'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd',
       'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd1', 'Rsvd2', 'Rsvd3', 'Rsvd4'],
      dtype='object')

In [15]:
df.columns

Index(['SYMBOL', 'SERIES', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'LAST', 'PREVCLOSE',
       'TOTTRDQTY', 'TOTTRDVAL', 'TIMESTAMP', 'TOTALTRADES', 'ISIN',
       'Unnamed: 13'],
      dtype='object')

In [ ]:
# Remove unwanted columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Column mapping
column_mapping = {
    "TckrSymb": "SYMBOL",
    "SctySrs": "SERIES",
    "OpnPric": "OPEN",
    "HghPric": "HIGH",
    "LwPric": "LOW",
    "ClsPric": "CLOSE",
    "LastPric": "LAST",
    "PrvsClsgPric": "PREVCLOSE",
    "TtlTradgVol": "TOTTRDQTY",
    "TtlTrfVal": "TOTTRDVAL",
    "BizDt": "TIMESTAMP",
    "TtlNbOfTxsExctd": "TOTALTRADES",
    "ISIN": "ISIN"
}

# Subset + rename
subset_full_data = full_data[list(column_mapping.keys())].copy()
subset_full_data.rename(columns=column_mapping, inplace=True)



In [18]:
subset_full_data.tail(2)

,SYMBOL,SERIES,OPEN,HIGH,LOW,CLOSE,LAST,PREVCLOSE,TOTTRDQTY,TOTTRDVAL,TIMESTAMP,TOTALTRADES,ISIN
1101373,ZYDUSLIFE,EQ,967.2,981.95,960.5,976.4,973.5,971.25,1188906,1.159735e+09,2025-07-25,39311,INE010B01027
1101374,ZYDUSWELL,EQ,2149.4,2157.40,2092.9,2102.6,2093.4,2138.10,44772,9.463579e+07,2025-07-25,8027,INE768C01010


In [22]:
# Concatenate vertically
combined_df = pd.concat(
    [df, subset_full_data],
    ignore_index=True
)

print(combined_df.shape)

(3610528, 13)


In [23]:
combined_df.shape

(3610528, 13)

In [24]:
import duckdb

# Name of database file you want to create
db_path = "All_equities_2020_onwards.duckdb"

# Connect (this creates the file if it doesn't exist)
with duckdb.connect(db_path) as con:
    con.execute("CREATE OR REPLACE TABLE equities AS SELECT * FROM combined_df")

print("DuckDB database created successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB database created successfully.


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
import shutil

source_path = "/content/All_equities_2020_onwards.duckdb"
destination_path = "/content/drive/MyDrive/All_equities_2020_onwards.duckdb"

shutil.copy(source_path, destination_path)

print("Database copied to Google Drive successfully.")

Database copied to Google Drive successfully.
